# Notebook 03 — EDA: ML Variant
**Question:** Is there any stable ML signal in the base-model data, and does it improve trading performance?

**Hypothesis (pre-registered):**  
With ~3,000 trades and expected AUC ≈ 0.56, we expect marginal but potentially real signal. The ML model's value should be measured by **trading uplift** (expectancy / drawdown when trading top-scored trades) rather than AUC alone. We expect the model to be calibrated but not dramatically better than the base.

**Threats to validity:** small sample, single fold temporal leakage risk, parameter tuning (tracked), permutation test validates signal existence.

In [2]:
import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

with open('../DATA/feature_cols.json') as f:
    RAW_FEATURE_COLS = json.load(f)

df = pd.read_csv('../DATA/base_ml_4yr_processed.csv', parse_dates=['open_time','close_time'])
df = df.sort_values('open_time').reset_index(drop=True)

print(f'Total trades: {len(df):,}')
print(f'Splits: {df["split"].value_counts().to_dict()}')

Total trades: 3,040
Splits: {'train': 1995, 'holdout': 549, 'val': 496}


## 1. Feature Engineering & Selection

In [4]:
# Select numeric, pre-entry features only
num_cols = df[RAW_FEATURE_COLS].select_dtypes(include=[np.number]).columns.tolist()

# Drop columns with >40% missing
miss_rate = df[num_cols].isnull().mean()
num_cols  = [c for c in num_cols if miss_rate[c] < 0.40]

TARGET = 'win'
df[TARGET] = (df['result_R'] > 0).astype(int)

print(f'Feature columns used: {len(num_cols)}')
print(num_cols)

Feature columns used: 53
['signal_type', 'open_hour', 'sl_pips', 'tp_pips', 'spread_pips', 'risk_pts', 'spread_pts', 'zone_age', 'zone_size_pips', 'zone_touches', 'body_size_ratio', 'engulf_depth', 'atr_m5_pips', 'atr_m15_pips', 'atr_h1_pips', 'atr_h4_pips', 'sl_to_atr_m15', 'sl_to_atr_h1', 'zone_to_atr_m15', 'atr_h1_pctile', 'day_range_to_adr', 'dist_ema50_h1_atr', 'dist_ema200_h1_atr', 'dist_ema50_h4_atr', 'dist_ema200_h4_atr', 'slope_ema50_h1_atr', 'slope_ema50_h4_atr', 'align_ema50_h1', 'align_ema50_h4', 'dist_pdh_R', 'dist_pdl_R', 'dist_dayhigh_R', 'dist_daylow_R', 'dist_round_R', 'ec_range_atr', 'ec_close_loc', 'ec_upper_wick', 'ec_lower_wick', 'ec_tickvol_ratio', 'rsi_m5', 'rsi_m15', 'mom5_atr', 'consec_dir_bars', 'mins_since_london', 'mins_since_ny', 'day_of_month', 'prev_R_1', 'prev_R_2', 'prev_R_3', 'trades_today_before', 'open_pos_other', 'mins_since_last_entry', 'streak_losses']


## 2. Time-Based Train / Val / Holdout Split

In [6]:
train_df   = df[df['split'] == 'train'].copy()
val_df     = df[df['split'] == 'val'].copy()
holdout_df = df[df['split'] == 'holdout'].copy()

def prep(df, cols, fit_scaler=None):
    X = df[cols].copy()
    # Impute with median
    for c in cols:
        X[c] = X[c].fillna(X[c].median())
    if fit_scaler is None:
        sc = StandardScaler().fit(X)
    else:
        sc = fit_scaler
    return sc.transform(X), sc

X_train, scaler = prep(train_df, num_cols)
X_val,   _      = prep(val_df,   num_cols, scaler)
X_hold,  _      = prep(holdout_df, num_cols, scaler)

y_train = train_df[TARGET].values
y_val   = val_df[TARGET].values
y_hold  = holdout_df[TARGET].values

print(f'Train   : {len(X_train):,}  |  Win rate: {y_train.mean():.3f}')
print(f'Val     : {len(X_val):,}  |  Win rate: {y_val.mean():.3f}')
print(f'Holdout : {len(X_hold):,}  |  Win rate: {y_hold.mean():.3f}')

Train   : 1,995  |  Win rate: 0.349
Val     : 496  |  Win rate: 0.419
Holdout : 549  |  Win rate: 0.348


## 3. Per-Feature Signal — Univariate AUC & Mutual Information

In [8]:
uni_auc = []
for i, col in enumerate(num_cols):
    try:
        auc = roc_auc_score(y_train, X_train[:, i])
        auc = max(auc, 1 - auc)  # directional
    except:
        auc = 0.5
    uni_auc.append(auc)

mi = mutual_info_classif(X_train, y_train, random_state=RANDOM_SEED)

feat_signal = pd.DataFrame({'feature': num_cols, 'uni_auc': uni_auc, 'mutual_info': mi})
feat_signal = feat_signal.sort_values('uni_auc', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top20 = feat_signal.head(20)

axes[0].barh(top20['feature'][::-1], top20['uni_auc'][::-1], color='steelblue', alpha=0.85)
axes[0].axvline(0.5, color='k', ls='--', lw=1)
axes[0].set_title('Top 20 Features — Univariate AUC')
axes[0].set_xlabel('AUC (max of direction)')

top20_mi = feat_signal.sort_values('mutual_info', ascending=False).head(20)
axes[1].barh(top20_mi['feature'][::-1], top20_mi['mutual_info'][::-1], color='darkorange', alpha=0.85)
axes[1].set_title('Top 20 Features — Mutual Information')
axes[1].set_xlabel('Mutual Information')

plt.tight_layout()
plt.savefig('../DATA/03_feature_signal.png', dpi=120)
plt.show()

display(feat_signal.head(20))

              feature   uni_auc  mutual_info
0         atr_h4_pips  0.542349     0.136353
1        day_of_month  0.537235     0.000000
2       ec_lower_wick  0.536633     0.016225
3      align_ema50_h4  0.536573     0.015240
4      align_ema50_h1  0.533769     0.009959
5         atr_m5_pips  0.531791     0.044063
6       mins_since_ny  0.530576     0.023346
7              rsi_m5  0.530306     0.066176
8    ec_tickvol_ratio  0.528644     0.045014
9   mins_since_london  0.526278     0.029951
10       dist_round_R  0.525835     0.044344
11       atr_m15_pips  0.525524     0.050678
12         dist_pdh_R  0.525489     0.032062
13         spread_pts  0.524005     0.001269
14        spread_pips  0.524005     0.000000
15       ec_close_loc  0.523559     0.028855
16     open_pos_other  0.523230     0.003939
17     dist_dayhigh_R  0.522803     0.048274
18        signal_type  0.520563     0.011384
19   day_range_to_adr  0.518465     0.106710


## 4. Feature Health — Correlation & Drift

In [10]:
# Correlation heatmap of top features
top_feats = feat_signal.head(15)['feature'].tolist()
corr_mat  = train_df[top_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax)
ax.set_title('Feature Correlation (top 15 by AUC, train set)')
plt.tight_layout()
plt.savefig('../DATA/03_feature_correlation.png', dpi=120)
plt.show()

In [11]:
# Feature drift: compare mean of top 5 features across splits
top5 = feat_signal.head(5)['feature'].tolist()
drift = df.groupby('split')[top5].mean().T
print('Feature means by split (drift check):')
display(drift.round(4))

Feature means by split (drift check):
split             holdout      train        val
atr_h4_pips     4437.0003  1213.0481  2522.2331
day_of_month      14.6867    15.5659    13.4919
ec_lower_wick      0.1734     0.1419     0.1493
align_ema50_h4     0.5519     0.6010     0.7702
align_ema50_h1     0.3953     0.4155     0.5565


## 5. XGBoost Model Training (Time-Split)


In [13]:
# Prepare raw (unscaled) for XGBoost
def prep_raw(df, cols):
    X = df[cols].copy()
    for c in cols:
        X[c] = X[c].fillna(X[c].median())
    return X.values

X_tr_raw  = prep_raw(train_df,   num_cols)
X_val_raw = prep_raw(val_df,     num_cols)
X_ho_raw  = prep_raw(holdout_df, num_cols)

dtrain = xgb.DMatrix(X_tr_raw,  label=y_train, feature_names=num_cols)
dval   = xgb.DMatrix(X_val_raw, label=y_val,   feature_names=num_cols)
dhold  = xgb.DMatrix(X_ho_raw,  label=y_hold,  feature_names=num_cols)

params = dict(
    objective        = 'binary:logistic',
    eval_metric      = 'auc',
    max_depth        = 5,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 20,
    seed             = RANDOM_SEED,
)

evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(params, dtrain, num_boost_round=500,
                  evals=evals, early_stopping_rounds=50,
                  verbose_eval=False)

prob_val  = model.predict(dval)
prob_hold = model.predict(dhold)

auc_val  = roc_auc_score(y_val,  prob_val)
auc_hold = roc_auc_score(y_hold, prob_hold)
print(f'Val AUC     : {auc_val:.4f}')
print(f'Holdout AUC : {auc_hold:.4f}')


Val AUC     : 0.5011
Holdout AUC : 0.5168


## 6. Fold-by-Fold AUC (Walk-Forward)

In [15]:
# 5 expanding-window folds on train+val
tv_df = df[df['split'].isin(['train','val'])].sort_values('open_time').reset_index(drop=True)
n     = len(tv_df)
fold_size = n // 5

best_round = model.best_iteration if hasattr(model, 'best_iteration') and model.best_iteration > 0 else 100

fold_aucs = []
for fold in range(1, 5):
    split_pt = fold * fold_size
    tr = tv_df.iloc[:split_pt]
    te = tv_df.iloc[split_pt:split_pt + fold_size]
    if len(te) < 30:
        continue

    X_tr = prep_raw(tr, num_cols)
    X_te = prep_raw(te, num_cols)
    y_tr = (tr['result_R'] > 0).values
    y_te = (te['result_R'] > 0).values

    dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=num_cols)
    dte = xgb.DMatrix(X_te, label=y_te, feature_names=num_cols)

    fold_model = xgb.train(params, dtr, num_boost_round=best_round, verbose_eval=False)
    probs = fold_model.predict(dte)
    auc   = roc_auc_score(y_te, probs) if len(np.unique(y_te)) > 1 else np.nan
    fold_aucs.append(dict(fold=fold, train_n=len(tr), test_n=len(te), auc=auc,
                          date_from=te['open_time'].min().date(),
                          date_to  =te['open_time'].max().date()))

fold_df = pd.DataFrame(fold_aucs)
display(fold_df.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(fold_df['fold'].astype(str), fold_df['auc'], color='steelblue', alpha=0.85)
ax.axhline(0.5, color='crimson', ls='--', lw=1.5, label='Random (AUC=0.5)')
ax.set_title('Walk-Forward Fold AUC'); ax.set_xlabel('Fold'); ax.set_ylabel('AUC')
ax.legend()
plt.tight_layout()
plt.savefig('../DATA/03_fold_auc.png', dpi=120)
plt.show()


   fold  train_n  test_n     auc   date_from     date_to
0     1      498     498  0.4799  2023-01-05  2024-02-05
1     2      996     498  0.5108  2024-02-05  2024-11-07
2     3     1494     498  0.5378  2024-11-08  2025-05-30
3     4     1992     498  0.4950  2025-05-30  2025-11-28


## 7. Calibration & Lift by Decile

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration
frac_pos, mean_pred = calibration_curve(y_val, prob_val, n_bins=10)
axes[0].plot(mean_pred, frac_pos, 'o-', color='steelblue', label='XGBoost (val)')
axes[0].plot([0,1],[0,1],'k--', lw=0.8, label='Perfect')
axes[0].set_title('Calibration Curve (Val)')
axes[0].set_xlabel('Mean Predicted Prob'); axes[0].set_ylabel('Fraction Positive')
axes[0].legend()

# Lift by decile
val_df2 = val_df.copy()
val_df2['prob'] = prob_val
val_df2['decile'] = pd.qcut(val_df2['prob'], 10, labels=False)
lift = val_df2.groupby('decile').agg(
    win_rate=('win','mean'),
    exp_r   =('result_R','mean'),
    n       =('result_R','count')
)
axes[1].bar(lift.index, lift['exp_r'], color=['crimson' if v<0 else 'steelblue' for v in lift['exp_r']])
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_title('Expectancy by Score Decile (Val, 0=lowest)')
axes[1].set_xlabel('Score Decile'); axes[1].set_ylabel('Avg R')

plt.tight_layout()
plt.savefig('../DATA/03_calibration_lift.png', dpi=120)
plt.show()

display(lift.round(4))

        win_rate   exp_r   n
decile                      
0         0.4600  0.3800  50
1         0.4000  0.2000  50
2         0.4490  0.3409  49
3         0.4000  0.2017  50
4         0.3673  0.0798  49
5         0.4800  0.4303  50
6         0.3061 -0.0515  49
7         0.3800  0.1398  50
8         0.4694  0.3918  49
9         0.4800  0.4402  50


## 8. Permutation Test — Beat Random Baseline

In [19]:
np.random.seed(RANDOM_SEED)
N_PERM = 500
observed_auc = auc_val

perm_aucs = []
for _ in range(N_PERM):
    y_shuf = np.random.permutation(y_val)
    perm_aucs.append(roc_auc_score(y_shuf, prob_val))

perm_aucs = np.array(perm_aucs)
p_val = (perm_aucs >= observed_auc).mean()

print(f'Observed AUC (val)      : {observed_auc:.4f}')
print(f'Permutation AUC mean    : {perm_aucs.mean():.4f} ± {perm_aucs.std():.4f}')
print(f'Permutation p-value     : {p_val:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(perm_aucs, bins=40, color='lightsteelblue', edgecolor='white', alpha=0.9)
ax.axvline(observed_auc, color='crimson', lw=2, ls='--', label=f'Observed AUC={observed_auc:.4f}')
ax.set_title('Permutation Test: AUC vs Shuffled Labels')
ax.set_xlabel('AUC'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../DATA/03_permutation_test.png', dpi=120)
plt.show()

Observed AUC (val)      : 0.5011
Permutation AUC mean    : 0.5017 ± 0.0250
Permutation p-value     : 0.5000


## 9. Trading Uplift — Top-Scored Trades vs Base

In [21]:
# Assign scores to ALL trades (using the full model)
X_all_raw = prep_raw(df, num_cols)
dall = xgb.DMatrix(X_all_raw, feature_names=num_cols)
df['ml_score'] = model.predict(dall)

def compute_metrics_df(dset, label):
    if len(dset) == 0:
        return dict(label=label, n=0, exp_r=np.nan, pf=np.nan, mdd=np.nan)
    exp_r   = dset['result_R'].mean()
    winners = dset.loc[dset['result_R']>0,'result_R'].sum()
    losers  = dset.loc[dset['result_R']<0,'result_R'].sum()
    pf      = winners/abs(losers) if losers != 0 else np.nan
    eq      = dset['result_R'].cumsum()
    mdd     = (eq - eq.cummax()).min()
    return dict(label=label, n=len(dset), exp_r=round(exp_r,4),
                pf=round(pf,3), mdd=round(mdd,3))

# Use holdout only (never seen by model training)
ho = holdout_df.copy()
ho['ml_score'] = model.predict(dhold)

results_uplift = []
for top_pct in [0.3, 0.4, 0.5, 0.6, 0.7, 1.0]:
    cutoff  = ho['ml_score'].quantile(1 - top_pct)
    top_set = ho[ho['ml_score'] >= cutoff]
    results_uplift.append(compute_metrics_df(top_set, f'Top {int(top_pct*100)}%'))

uplift_df = pd.DataFrame(results_uplift).set_index('label')
print('ML Uplift — Holdout (top X% of scored trades):')
display(uplift_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in [(axes[0],'exp_r','Expectancy (R)'),
                           (axes[1],'pf','Profit Factor'),
                           (axes[2],'mdd','Max Drawdown (R)')]:
    vals = uplift_df[metric]
    ax.bar(vals.index, vals.values,
           color=['steelblue' if v > 0 else 'crimson' for v in vals.values])
    ax.set_title(title); ax.set_xlabel('Top % Scored')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('ML Uplift by Score Threshold (Holdout)', y=1.02)
plt.tight_layout()
plt.savefig('../DATA/03_ml_uplift.png', dpi=120)
plt.show()


ML Uplift — Holdout (top X% of scored trades):
            n   exp_r     pf     mdd
label                               
Top 30%   165  0.1046  1.164 -24.497
Top 40%   220  0.0596  1.092 -30.266
Top 50%   275  0.1011  1.157 -23.580
Top 60%   329  0.0751  1.115 -27.666
Top 70%   384  0.0429  1.065 -37.700
Top 100%  549  0.0336  1.050 -42.972


## 10. Feature Importance

In [23]:
score_gain = model.get_score(importance_type='gain')
score_weight = model.get_score(importance_type='weight')

importance = pd.DataFrame({
    'feature'    : num_cols,
    'gain'       : [score_gain.get(c, 0.0) for c in num_cols],
    'split_count': [score_weight.get(c, 0) for c in num_cols]
}).sort_values('gain', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))
top_imp = importance.head(20)
ax.barh(top_imp['feature'][::-1], top_imp['gain'][::-1], color='steelblue', alpha=0.85)
ax.set_title('XGBoost Feature Importance (Gain) — Top 20')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig('../DATA/03_feature_importance.png', dpi=120)
plt.show()

display(importance.head(20))

# Save model probabilities for notebook 04
df[['pos_id','open_time','ml_score']].to_csv('../DATA/ml_scores_all.csv', index=False)
print('\nSaved ml_scores_all.csv')


               feature       gain  split_count
0       align_ema50_h4  12.379592         11.0
1       dist_dayhigh_R  10.889157          9.0
2    dist_ema50_h4_atr  10.383261         12.0
3         day_of_month  10.290640         20.0
4       align_ema50_h1  10.018966          3.0
5         ec_close_loc   9.907712         12.0
6          atr_h4_pips   9.838625         39.0
7         atr_m15_pips   9.775971          8.0
8   dist_ema200_h1_atr   9.730032         13.0
9     day_range_to_adr   9.581947         23.0
10  slope_ema50_h4_atr   9.545759         13.0
11  slope_ema50_h1_atr   9.467364         10.0
12            prev_R_1   9.400431          9.0
13           open_hour   9.330231          8.0
14             sl_pips   9.181108          5.0
15             rsi_m15   9.163361         18.0
16       dist_daylow_R   8.793947          4.0
17         atr_m5_pips   8.672190         15.0
18       streak_losses   8.660941          5.0
19        sl_to_atr_h1   8.624306          6.0

Saved ml_sco